In [4]:
!pip install cantera
!pip install CoolProp
import cantera as ct
print(ct.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 95.0 MB/s eta 0:00:00
3.2.0


In [9]:
import os
import pandas as pd
from CoolProp.CoolProp import PropsSI

methane_mass_fraction = 0.79 # BOG용: 메탄 79%
nitrogen_mass_fraction = 0.21 # BOG용: 질소 21%
FLUID = f"Methane[{methane_mass_fraction}]&Nitrogen[{nitrogen_mass_fraction}]" # BOG용 유체

methane_mass_fraction_lng = 1.0 # LNG용: 메탄 100%
nitrogen_mass_fraction_lng = 0.0 # LNG용: 질소 0%
FLUID_LNG = f"Methane[{methane_mass_fraction_lng}]&Nitrogen[{nitrogen_mass_fraction_lng}]" # LNG용 유체

def C_to_K(T_C):
    # 섭씨 온도를 켈빈 온도로 변환
    return T_C + 273.15

def K_to_C(T_K):
    # 켈빈 온도를 섭씨 온도로 변환
    return T_K - 273.15

def bar_to_Pa(P_bar):
    # bar 단위를 파스칼(Pa)로 변환
    return P_bar * 1e5

def Pa_to_bar(P_Pa):
    # 파스칼(Pa) 단위를 bar로 변환
    return P_Pa / 1e5

# =====================================================
# 1. CoolProp 속성 도우미 함수들
# =====================================================

def h_TP(T, P, fluid):
    # 온도(T)와 압력(P)으로 질량 엔탈피(Hmass) 계산
    try:
        return PropsSI("Hmass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Hmass", "T", T + 0.1, "P", P, fluid)

def s_TP(T, P, fluid):
    # 온도(T)와 압력(P)으로 질량 엔트로피(Smass) 계산
    try:
        return PropsSI("Smass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Smass", "T", T + 0.1, "P", P, fluid)

def T_Ph(P, h, fluid):
    # 압력(P)과 질량 엔탈피(h)로 온도(T) 계산
    return PropsSI("T", "P", P, "Hmass", h, fluid)

def h_PS(P, s, fluid):
    # 압력(P)과 질량 엔트로피(s)로 질량 엔탈피(Hmass) 계산 (등엔트로피 과정)
    return PropsSI("Hmass", "P", P, "Smass", s, fluid)

def rho_TP(T, P, fluid):
    # 온도(T)와 압력(P)으로 질량 밀도(Dmass) 계산
    try:
        return PropsSI("Dmass", "T", T, "P", P, fluid)
    except ValueError:
        # 오류 발생 시 약간의 온도 보정
        return PropsSI("Dmass", "T", T + 0.1, "P", P, fluid)

# =====================================================
# 2. 컴포넌트 모델
# =====================================================

def liquid_pump(m_dot, T_in, P_in, P_out, eta=0.75, fluid=FLUID_LNG):
    # 액체 펌프 모델
    h_in = h_TP(T_in, P_in, fluid)
    rho = rho_TP(T_in, P_in, fluid)

    if rho < 100.0:
        # 펌프 입구 밀도가 너무 낮으면 오류 발생 (액체가 아닌 증기/이상 상태일 가능성)
        raise ValueError(
            f"액체 펌프 입구 밀도가 너무 낮습니다 ({rho:.3f} kg/m3). "
            "입구는 증기/이상 상태일 가능성이 높습니다. "
            "펌프 입구 압력을 높이거나 펌프 입구 온도를 낮추십시오."
        )

    # 펌프 일 (엔탈피 변화)
    dh = (P_out - P_in) / rho / eta
    h_out = h_in + dh
    T_out = T_Ph(P_out, h_out, fluid)
    W = m_dot * dh

    return {
        "T_out": T_out, # 출구 온도
        "P_out": P_out, # 출구 압력
        "h_out": h_out, # 출구 엔탈피
        "W": W,         # 펌프 동력
        "rho_in": rho,  # 입구 밀도
    }

def compressor(m_dot, T_in, P_in, P_out, eta=0.75, fluid=FLUID):
    # 압축기 모델
    if m_dot <= 0:
        # 질량 유량이 없으면 압축기 동력은 0
        return {
            "T_out": T_in,
            "P_out": P_out,
            "h_out": h_TP(T_in, P_out, fluid),
            "W": 0.0,
        }

    h_in = h_TP(T_in, P_in, fluid)
    s_in = s_TP(T_in, P_in, fluid)
    # 등엔트로피 출구 엔탈피
    h_out_s = h_PS(P_out, s_in, fluid)
    # 실제 출구 엔탈피 (효율 반영)
    h_out = h_in + (h_out_s - h_in) / eta
    T_out = T_Ph(P_out, h_out, fluid)
    # 압축기 동력
    W = m_dot * (h_out - h_in)

    return {
        "T_out": T_out,
        "P_out": P_out,
        "h_out": h_out,
        "W": W,
    }

def heat_exchanger_effectiveness(
    m_cold,
    T_cold_in,
    P_cold,
    fluid_cold,
    m_hot,
    T_hot_in,
    P_hot,
    fluid_hot,
    effectiveness=0.60,
    min_delta_T=3.0,
):
    """
    일반적인 열교환기 유용도(effectiveness) 모델.
    뜨거운 스트림이 차가운 스트림에 열을 전달합니다.
    """
    if m_hot <= 0 or effectiveness <= 0:
        # 뜨거운 유량 또는 유용도가 0 이하면 열 전달 없음
        return {
            "Q": 0.0,
            "T_cold_out": T_cold_in,
            "h_cold_out": h_TP(T_cold_in, P_cold, fluid_cold),
            "T_hot_out": T_hot_in,
            "h_hot_out": h_TP(T_hot_in, P_hot, fluid_hot),
        }

    h_hot_in = h_TP(T_hot_in, P_hot, fluid_hot)
    h_cold_in = h_TP(T_cold_in, P_cold, fluid_cold)
    # 최소 접근 온도 차이를 고려한 뜨거운 스트림의 최소 온도
    T_hot_min = T_cold_in + min_delta_T
    h_hot_min = h_TP(T_hot_min, P_hot, fluid_hot)

    # 최대 가능한 열량 (뜨거운 스트림 기준)
    Q_max_hot = m_hot * max(h_hot_in - h_hot_min, 0.0)
    # 실제 열 전달량 (유용도 반영)
    Q = effectiveness * Q_max_hot

    # 출구 엔탈피 계산
    h_hot_out = h_hot_in - Q / m_hot
    h_cold_out = h_cold_in + Q / m_cold

    # 출구 온도 계산
    T_hot_out = T_Ph(P_hot, h_hot_out, fluid_hot)
    T_cold_out = T_Ph(P_cold, h_cold_out, fluid_cold)

    return {
        "Q": Q,                             # 열 전달량
        "T_cold_out": T_cold_out,         # 차가운 스트림 출구 온도
        "h_cold_out": h_cold_out,         # 차가운 스트림 출구 엔탈피
        "T_hot_out": T_hot_out,           # 뜨거운 스트림 출구 온도
        "h_hot_out": h_hot_out,           # 뜨거운 스트림 출구 엔탈피
    }

def set_stream_temperature(m_dot, T_in, P, T_target, fluid):
    """
    히터/쿨러를 적용하여 동일 압력에서 스트림 온도를 설정합니다.
    양의 Q는 스트림에 열이 추가됨을 의미합니다.
    """
    h_in = h_TP(T_in, P, fluid)
    h_out = h_TP(T_target, P, fluid)
    # 필요한 열량
    Q = m_dot * (h_out - h_in)

    return {
        "Q": Q,
        "T_out": T_target,
        "P_out": P,
        "h_out": h_out,
    }


# =====================================================
# 3. 다단 HPC 모델 (사용하지 않음)
# =====================================================

def get_hpc_stage_pressures(P_in, P_out, n_stages):
    """
    균등 압력비 단계.
    경계 압력: [P0, P1, ..., Pn]을 반환합니다.
    """
    pr_stage = (P_out / P_in) ** (1.0 / n_stages)
    pressures = [P_in]
    for _ in range(n_stages):
        pressures.append(pressures[-1] * pr_stage)
    pressures[-1] = P_out # 마지막 압력을 P_out으로 강제 설정하여 부동 소수점 오차 방지
    return pressures


def multistage_hpc_no_intercooling(m_dot, T_in, P_in, P_out, eta=0.75, n_stages=3, fluid=FLUID):
    # 중간 냉각 없는 다단 고압 압축기(HPC) 모델
    # 이 함수는 현재 간소화된 모델에서 사용되지 않습니다.
    pressures = get_hpc_stage_pressures(P_in, P_out, n_stages)
    T = T_in
    W_total = 0.0
    stage_results = []

    for i in range(n_stages):
        T_stage_in = T
        comp = compressor(m_dot, T_stage_in, pressures[i], pressures[i + 1], eta, fluid=fluid)
        W_total += comp["W"]
        T = comp["T_out"]

        stage_results.append({
            "stage": i + 1,
            "P_in": pressures[i],
            "P_out": pressures[i + 1],
            "T_in": T_stage_in,
            "T_out": comp["T_out"],
            "W": comp["W"],
        })

    return {
        "T_out": T,
        "P_out": P_out,
        "W": W_total,
        "stages": stage_results,
    }


# =====================================================
# 4. 전체 공정 모델 헬퍼 함수
# =====================================================

def _assemble_simplified_results(
    params,
    m_lng,
    m_bog_total,
    vol_lng_m3_per_hr,
    rho_lng_tank,
    vol_bog_m3_per_hr,
    rho_bog_tank,
    pump_results,
    Q_lng_pump_out_heating,
    T_lng_after_pump_calc,
    P_lng_after_pump,
    T_lng_after_desuperheater,
    T_bog_after_desuperheater,
    Q_desuperheater,
    comp_bog_results,
    T_bog_comp_out,
    P_bog_comp_out
):

    # 총 전력 및 열량 계산
    total_power_mw = (pump_results["W"] + comp_bog_results["W"]) / 1e6
    pump_out_heating_duty_mw = Q_lng_pump_out_heating / 1e6
    desuperheater_duty_mw = Q_desuperheater / 1e6

    return {
        "LNG_TANK_VOL_FLOW_M3_PER_HR": vol_lng_m3_per_hr, # LNG 탱크 체적 유량 (입력)
        "LNG_TANK_RHO_KG_PER_M3": rho_lng_tank, # LNG 탱크 밀도 (계산)
        "LNG_MASS_FLOW_KG_PER_S": m_lng, # LNG 질량 유량 (계산)

        "BOG_TANK_VOL_FLOW_M3_PER_HR": vol_bog_m3_per_hr, # BOG 탱크 체적 유량 (입력)
        "BOG_TANK_RHO_KG_PER_M3": rho_bog_tank, # BOG 탱크 밀도 (계산)
        "BOG_TOTAL_MASS_FLOW_KG_PER_S": m_bog_total, # BOG 질량 유량 (계산)

        "LNG_TANK_T_C": K_to_C(params["T_lng_tank"]), # LNG 탱크 온도 (섭씨)
        "LNG_TANK_P_BAR": Pa_to_bar(params["P_lng_tank"]), # LNG 탱크 압력 (bar)
        "LNG_PUMP_OUT_CALC_T_C": K_to_C(T_lng_after_pump_calc), # LNG 펌프 출구 계산된 온도 (섭씨)
        "LNG_PUMP_OUT_P_BAR": Pa_to_bar(P_lng_after_pump), # LNG 펌프 출구 압력 (bar)
        "LNG_PUMP_OUT_HEATED_T_C": K_to_C(T_lng_after_desuperheater), # LNG 디슈퍼히터 출구 온도 (섭씨)
        "LNG_PUMP_OUT_HEATING_DUTY_MW": pump_out_heating_duty_mw, # LNG 펌프 출구 가열 열량 (MW)

        "BOG_TANK_T_C": K_to_C(params["T_bog_tank"]),     # BOG 탱크 온도 (섭씨)
        "BOG_TANK_P_BAR": Pa_to_bar(params["P_bog_tank"]), # BOG 탱크 압력 (bar)
        "BOG_DESUPERHEATER_OUT_T_C": K_to_C(T_bog_after_desuperheater), # BOG 디슈퍼히터 출구 온도 (섭씨)
        "BOG_COMP_OUT_T_C": K_to_C(T_bog_comp_out),       # BOG 압축기 출구 온도 (섭씨)
        "BOG_COMP_OUT_P_BAR": Pa_to_bar(P_bog_comp_out), # BOG 압축기 출구 압력 (bar)
        "BOG_FINAL_SENDOUT_MASS_FLOW": m_bog_total, # BOG 최종 송출 질량 유량 (압축기 출구) (계산)

        "DESUPERHEATER_DUTY_MW": desuperheater_duty_mw,    # 디슈퍼히터 열량 (MW)
        "LNG_PUMP_POWER_MW": pump_results["W"] / 1e6,              # LNG 펌프 동력 (MW)
        "BOG_COMP_POWER_MW": comp_bog_results["W"] / 1e6,          # BOG 압축기 동력 (MW)
        "TOTAL_POWER_MW": total_power_mw,                  # 총 동력 (MW)
    }

def simulate_process(params):
    # 주요 매개변수 추출
    v_lng_tank_m3_per_hr = params["v_lng_tank_m3_per_hr"]
    v_bog_tank_m3_per_hr = params["v_bog_tank_m3_per_hr"]

    # 명확성을 위한 주요 매개변수 추출
    P_lng_tank = params["P_lng_tank"]
    T_lng_tank = params["T_lng_tank"]
    P_lng_high = params["P_lng_high"]
    T_lng_pump_out_initial = params["T_lng_pump_out_heated"]

    P_bog_tank = params["P_bog_tank"]
    T_bog_tank = params["T_bog_tank"]
    P_bog_comp_out_target = params["P_bog_comp_out"]

    # LNG 탱크 밀도 계산 및 질량 유량 변환
    rho_lng_tank = rho_TP(T_lng_tank, P_lng_tank, FLUID_LNG)
    m_lng = v_lng_tank_m3_per_hr * rho_lng_tank / 3600.0 # m3/h * kg/m3 / 3600 s/h = kg/s

    # BOG 탱크 밀도 계산 및 질량 유량 변환
    rho_bog_tank = rho_TP(T_bog_tank, P_bog_tank, FLUID)
    m_bog_total = v_bog_tank_m3_per_hr * rho_bog_tank / 3600.0 # m3/h * kg/m3 / 3600 s/h = kg/s


    # 1. LNG 펌프
    pump = liquid_pump(
        m_dot=m_lng,
        T_in=T_lng_tank,
        P_in=P_lng_tank,
        P_out=P_lng_high,
        eta=params["pump_eta"],
        fluid=FLUID_LNG
    )

    T_lng_after_pump_calc = pump["T_out"]
    P_lng_after_pump = pump["P_out"]

    # 펌프 출구 초기 가열 (디슈퍼히터 이전)
    pump_out_heater = set_stream_temperature(
        m_dot=m_lng,
        T_in=T_lng_after_pump_calc,
        P=P_lng_after_pump,
        T_target=T_lng_pump_out_initial,
        fluid=FLUID_LNG
    )
    T_lng_initial_heated = pump_out_heater["T_out"]
    Q_lng_pump_out_heating = pump_out_heater["Q"]

    # 2. 디슈퍼히터 (LNG cold side, BOG hot side)
    # BOG는 탱크에서 바로 디슈퍼히터로 들어갑니다.
    desuperheater_results = heat_exchanger_effectiveness(
        m_cold=m_lng,
        T_cold_in=T_lng_initial_heated,
        P_cold=P_lng_after_pump,
        fluid_cold=FLUID_LNG,
        m_hot=m_bog_total,
        T_hot_in=T_bog_tank, # BOG는 탱크에서 직접 들어옴
        P_hot=P_bog_tank, # BOG는 탱크 압력으로 들어옴
        fluid_hot=FLUID,
        effectiveness=params["desuperheater_effectiveness"]
    )

    T_lng_after_desuperheater = desuperheater_results["T_cold_out"]
    T_bog_after_desuperheater = desuperheater_results["T_hot_out"]
    Q_desuperheater = desuperheater_results["Q"]

    # 3. BOG 저압 압축기 (디슈퍼히터 이후, 유일한 BOG 압축기)
    # BOG 공정은 여기서 검토 종료
    comp_bog_results = compressor(
        m_dot=m_bog_total,
        T_in=T_bog_after_desuperheater, # 디슈퍼히터 출구 온도가 압축기 입구 온도
        P_in=P_bog_tank, # 디슈퍼히터에서 압력 강하는 없다고 가정, 탱크 압력으로 압축기 입구
        P_out=P_bog_comp_out_target, # 목표 압축기 출구 압력
        eta=params["comp_eta"],
        fluid=FLUID
    )
    T_bog_comp_out = comp_bog_results["T_out"]
    P_bog_comp_out = comp_bog_results["P_out"]


    # 4. 결과 취합
    return _assemble_simplified_results(
        params=params,
        m_lng=m_lng,
        m_bog_total=m_bog_total,
        vol_lng_m3_per_hr=v_lng_tank_m3_per_hr,
        rho_lng_tank=rho_lng_tank,
        vol_bog_m3_per_hr=v_bog_tank_m3_per_hr,
        rho_bog_tank=rho_bog_tank,
        pump_results=pump,
        Q_lng_pump_out_heating=Q_lng_pump_out_heating,
        T_lng_after_pump_calc=T_lng_after_pump_calc,
        P_lng_after_pump=P_lng_after_pump,
        T_lng_after_desuperheater=T_lng_after_desuperheater,
        T_bog_after_desuperheater=T_bog_after_desuperheater,
        Q_desuperheater=Q_desuperheater,
        comp_bog_results=comp_bog_results,
        T_bog_comp_out=T_bog_comp_out,
        P_bog_comp_out=P_bog_comp_out
    )


In [7]:
import pandas as pd
from CoolProp.CoolProp import PropsSI

def get_phase_string(T_C, P_bar, fluid):
    T_K = C_to_K(T_C)
    P_Pa = bar_to_Pa(P_bar)
    try:
        raw_phase_output = PropsSI('Phase', 'T', T_K, 'P', P_Pa, fluid)

        phase_name_str = None

        if isinstance(raw_phase_output, (int, float)): # 정수 또는 실수 형태의 위상 코드를 처리
            phase_code = int(raw_phase_output) # 정수형으로 변환하여 매핑
            # CoolProp의 표준 위상 이름 문자열로 매핑합니다.
            if phase_code == 0:
                phase_name_str = 'undefined'
            elif phase_code == 1:
                phase_name_str = 'liquid'
            elif phase_code == 2:
                phase_name_str = 'gas'
            elif phase_code == 3:
                phase_name_str = 'supercritical'
            elif phase_code == 4:
                phase_name_str = 'supercritical_gas'
            elif phase_code == 5:
                phase_name_str = 'supercritical_liquid'
            elif phase_code == 6:
                phase_name_str = 'critical_point'
            elif phase_code == 7:
                phase_name_str = 'two_phase'
            else:
                phase_name_str = f'알 수 없는 위상 코드: {phase_code}'
        elif isinstance(raw_phase_output, str):
            phase_name_str = raw_phase_output
        else:
            # 예측하지 못한 다른 타입의 출력인 경우
            phase_name_str = '알 수 없음 (CoolProp 출력 형식 오류 - 타입 미확인)'

        # 매핑된 문자열 위상 이름을 한국어로 번역
        if phase_name_str == 'liquid':
            return '액체'
        elif phase_name_str == 'gas':
            return '기체'
        elif phase_name_str in ['supercritical', 'supercritical_gas', 'supercritical_liquid']:
            return '초임계' # 초임계 관련 위상들을 하나로 묶어 처리
        elif phase_name_str == 'critical_point':
            return '임계점'
        elif phase_name_str == 'two_phase':
            Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
            return f'기액 이중상 (건도: {Q:.2f})'
        elif phase_name_str == 'undefined':
            # 'undefined'인 경우, 가능하면 건도를 통해 추가 추정
            try:
                Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
                if Q is not None:
                    if Q <= 0: return '과냉각 액체 또는 포화 액체'
                    elif Q >= 1: return '과열 증기 또는 포화 증기'
                    else: return f'기액 이중상 (건도: {Q:.2f})'
            except ValueError:
                pass # Quality 계산 실패 시 무시
            return '정의되지 않음'
        else:
            return phase_name_str # CoolProp이 반환하는 기타 문자열 위상 이름

    except ValueError:
        # CoolProp 계산 오류 발생 시 (예: 정의되지 않은 영역), 건도를 통해 추가 추정
        try:
            Q = PropsSI('Q', 'T', T_K, 'P', P_Pa, fluid)
            if Q is not None:
                if Q <= 0: return '과냉각 액체 또는 포화 액체'
                elif Q >= 1: return '과열 증기 또는 포화 증기'
                else: return f'기액 이중상 (건도: {Q:.2f})'
        except Exception:
            pass # Quality 계산 실패 시 무시
        return '알 수 없음 (CoolProp 오류)'
    except Exception as e:
        return f'오류 발생: {type(e).__name__}'

# simulate_process 함수를 호출하여 results를 업데이트합니다.
results = simulate_process(params)

# 데이터 추출 및 가공
data_points = []

# LNG 조성 정보 문자열
composition_str_lng = f"메탄 {methane_mass_fraction_lng*100:.0f}%, 질소 {nitrogen_mass_fraction_lng*100:.0f}%"
# BOG 조성 정보 문자열
composition_str_bog = f"메탄 {methane_mass_fraction*100:.0f}%, 질소 {nitrogen_mass_fraction*100:.0f}%"

# --- LNG 스트림 ---
# LNG 탱크
data_points.append({
    "위치": "LNG 탱크",
    "온도 [°C]": f"{results["LNG_TANK_T_C"]:.2f} (입력)",
    "압력 [bar]": f"{results["LNG_TANK_P_BAR"]:.2f} (입력)",
    "체적 유량 [m³/h]": f"{results["LNG_TANK_VOL_FLOW_M3_PER_HR"]:.2f} (입력)",
    "밀도 [kg/m³]": f"{results["LNG_TANK_RHO_KG_PER_M3"]:.2f} (계산)",
    "질량 유량 [kg/s]": f"{results["LNG_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(results["LNG_TANK_T_C"], results["LNG_TANK_P_BAR"], fluid=FLUID_LNG),
    "조성": composition_str_lng,
    "다음 위치": "LNG 펌프 출구 (계산)"
})

# LNG 펌프 출구 (계산)
data_points.append({
    "위치": "LNG 펌프 출구 (계산)",
    "온도 [°C]": f"{results["LNG_PUMP_OUT_CALC_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["LNG_PUMP_OUT_P_BAR"]:.2f} (입력)",
    "체적 유량 [m³/h]": "N/A",
    "밀도 [kg/m³]": "N/A",
    "질량 유량 [kg/s]": f"{results["LNG_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(results["LNG_PUMP_OUT_CALC_T_C"], results["LNG_PUMP_OUT_P_BAR"], fluid=FLUID_LNG),
    "조성": composition_str_lng,
    "다음 위치": "LNG 펌프 출구 (초기 가열 목표)"
})

# LNG 펌프 출구 (초기 가열 목표)
data_points.append({
    "위치": "LNG 펌프 출구 (초기 가열 목표)",
    "온도 [°C]": f"{K_to_C(params["T_lng_pump_out_heated"]):.2f} (입력)",
    "압력 [bar]": f"{results["LNG_PUMP_OUT_P_BAR"]:.2f} (입력)",
    "체적 유량 [m³/h]": "N/A",
    "밀도 [kg/m³]": "N/A",
    "질량 유량 [kg/s]": f"{results["LNG_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(K_to_C(params["T_lng_pump_out_heated"]), results["LNG_PUMP_OUT_P_BAR"], fluid=FLUID_LNG),
    "조성": composition_str_lng,
    "다음 위치": "LNG 디슈퍼히터 출구"
})

# LNG 디슈퍼히터 출구
data_points.append({
    "위치": "LNG 디슈퍼히터 출구",
    "온도 [°C]": f"{results["LNG_PUMP_OUT_HEATED_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["LNG_PUMP_OUT_P_BAR"]:.2f} (입력)",
    "체적 유량 [m³/h]": "N/A",
    "밀도 [kg/m³]": "N/A",
    "질량 유량 [kg/s]": f"{results["LNG_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(results["LNG_PUMP_OUT_HEATED_T_C"], results["LNG_PUMP_OUT_P_BAR"], fluid=FLUID_LNG),
    "조성": composition_str_lng,
    "다음 위치": "(LNG 최종 스코프)"
})

# --- BOG 스트림 ---
# BOG 탱크
data_points.append({
    "위치": "BOG 탱크",
    "온도 [°C]": f"{results["BOG_TANK_T_C"]:.2f} (입력)",
    "압력 [bar]": f"{results["BOG_TANK_P_BAR"]:.2f} (입력)",
    "체적 유량 [m³/h]": f"{results["BOG_TANK_VOL_FLOW_M3_PER_HR"]:.2f} (입력)",
    "밀도 [kg/m³]": f"{results["BOG_TANK_RHO_KG_PER_M3"]:.2f} (계산)",
    "질량 유량 [kg/s]": f"{results["BOG_TOTAL_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_TANK_T_C"], results["BOG_TANK_P_BAR"], fluid=FLUID),
    "조성": composition_str_bog,
    "다음 위치": "BOG 디슈퍼히터 출구"
})

# BOG 디슈퍼히터 출구
data_points.append({
    "위치": "BOG 디슈퍼히터 출구",
    "온도 [°C]": f"{results["BOG_DESUPERHEATER_OUT_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["BOG_TANK_P_BAR"]:.2f} (입력)", # 디슈퍼히터에서 BOG 압력 변화는 없다고 가정
    "체적 유량 [m³/h]": "N/A",
    "밀도 [kg/m³]": "N/A",
    "질량 유량 [kg/s]": f"{results["BOG_TOTAL_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_DESUPERHEATER_OUT_T_C"], results["BOG_TANK_P_BAR"], fluid=FLUID),
    "조성": composition_str_bog,
    "다음 위치": "BOG 저압 압축기 출구"
})

# BOG 저압 압축기 출구
data_points.append({
    "위치": "BOG 저압 압축기 출구",
    "온도 [°C]": f"{results["BOG_COMP_OUT_T_C"]:.2f} (계산)",
    "압력 [bar]": f"{results["BOG_COMP_OUT_P_BAR"]:.2f} (입력)",
    "체적 유량 [m³/h]": "N/A",
    "밀도 [kg/m³]": "N/A",
    "질량 유량 [kg/s]": f"{results["BOG_TOTAL_MASS_FLOW_KG_PER_S"]:.2f} (계산)",
    "위상": get_phase_string(results["BOG_COMP_OUT_T_C"], results["BOG_COMP_OUT_P_BAR"], fluid=FLUID),
    "조성": composition_str_bog,
    "다음 위치": "(BOG 최종 스코프)"
})

df_streams_full = pd.DataFrame(data_points)

print("\n=== 간소화된 시스템 스트림 데이터 요약 ===")
display(df_streams_full.round(2).fillna('N/A'))


=== 간소화된 시스템 스트림 데이터 요약 ===


,위치,온도 [°C],압력 [bar],질량 유량 [kg/s],위상,조성,다음 위치
0,LNG 탱크,-160.00 (입력),5.00 (입력),65.00 (입력),과냉각 액체 또는 포화 액체,"메탄 100%, 질소 0%",LNG 펌프 출구 (계산)
1,LNG 펌프 출구 (계산),-156.27 (계산),80.00 (입력),65.00 (입력),초임계,"메탄 100%, 질소 0%",LNG 펌프 출구 (초기 가열 목표)
2,LNG 펌프 출구 (초기 가열 목표),-130.00 (입력),80.00 (입력),65.00 (입력),초임계,"메탄 100%, 질소 0%",LNG 디슈퍼히터 출구
3,LNG 디슈퍼히터 출구,-125.04 (계산),80.00 (입력),65.00 (입력),초임계,"메탄 100%, 질소 0%",(LNG 최종 스코프)
4,BOG 탱크,-100.00 (입력),1.20 (입력),35.00 (입력),초임계,"메탄 79%, 질소 21%",BOG 디슈퍼히터 출구
5,BOG 디슈퍼히터 출구,-118.93 (계산),1.20 (입력),35.00 (입력),초임계,"메탄 79%, 질소 21%",BOG 저압 압축기 출구
6,BOG 저압 압축기 출구,26.90 (계산),10.00 (입력),35.00 (계산),초임계,"메탄 79%, 질소 21%",(BOG 최종 스코프)


In [8]:
params = {
    "v_lng_tank_m3_per_hr": 100.0, # LNG 체적 유량 (m3/h) - 입력
    "v_bog_tank_m3_per_hr": 150.0, # BOG 체적 유량 (m3/h) - 입력

    # 총 BOG 유량, BOG 압축기 후 분할 (이제는 체적 유량으로부터 계산됩니다.)
    # "m_bog_total": 35.0,

    # LNG 저장 탱크 및 실제 펌프 입구 조건
    "T_lng_tank": C_to_K(-162.0),
    "P_lng_tank": bar_to_Pa(5.0),

    # LNG 2차 펌프 출구는 펌핑 후 가열된다고 가정합니다.
    # 이것은 LNG 스트림의 디슈퍼히터 이전의 목표 온도입니다.
    "T_lng_pump_out_heated": C_to_K(-130.0),

    # LNG 2차 펌프 출구 압력
    "P_lng_high": bar_to_Pa(80.0),

    "T_bog_tank": C_to_K(-100.0),
    "P_bog_tank": bar_to_Pa(1.2),
    "P_bog_comp_out": bar_to_Pa(10.0), # 단일 BOG 압축기의 목표 출구 압력

    # 다음 매개변수들은 더 이상 사용되지 않습니다.
    # "P_bog_hpc_out": bar_to_Pa(80.0),
    # "T_sendout": C_to_K(15.0),
    # "P_sendout": bar_to_Pa(70.0),

    "pump_eta": 0.75,
    "comp_eta": 0.75,
    "desuperheater_effectiveness": 0.70, # 디슈퍼히터 효율에 대한 새로운 매개변수
}

In [74]:
mermaid_diagram_code = """graph TD
    subgraph LNG Stream
        A[LNG 탱크] --> B(LNG 펌프)
        B --> C(LNG 프리히터)
        C --> D{디슈퍼히터 (LNG 냉각 측)}
    end

    subgraph BOG Stream
        E[BOG 탱크] --> F{디슈퍼히터 (BOG 가열 측)}
        F --> G(BOG 저압 압축기)
    end

    D -- 냉각된 LNG --> H[LNG 검토 종료]
    G -- 압축된 BOG --> I[BOG 검토 종료]

    style A fill:#f9f,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333,stroke-width:2px
    style H fill:#f9f,stroke:#333,stroke-width:2px
    style I fill:#f9f,stroke:#333,stroke-width:2px
"""

print("Mermaid diagram code generated.")

Mermaid diagram code generated.


## 간소화된 LNG 기화 공정 시스템 구성도

```mermaid
graph TD
    subgraph LNG Stream
        A[LNG 탱크] --> B(LNG 펌프)
        B --> C(LNG 프리히터)
        C --> D{디슈퍼히터 (LNG 냉각 측)}
    end

    subgraph BOG Stream
        E[BOG 탱크] --> F{디슈퍼히터 (BOG 가열 측)}
        F --> G(BOG 저압 압축기)
    end

    D -- 냉각된 LNG --> H[LNG 검토 종료]
    G -- 압축된 BOG --> I[BOG 검토 종료]

    style A fill:#f9f,stroke:#333,stroke-width:2px
    style E fill:#f9f,stroke:#333,stroke-width:2px
    style H fill:#f9f,stroke:#333,stroke-width:2px
    style I fill:#f9f,stroke:#333,stroke-width:2px
```
